<a href="https://colab.research.google.com/github/Insight-Sogang-Univ/insight-15th/blob/junmo/advanced/junmo/session05/%EC%83%9D%EC%84%B1%EB%AA%A8%EB%8D%B8%EC%82%AC%EC%A0%84%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 생성 모델과 AI 최적화

## 1. 생성 모델

### 1-1. 분류 모델 vs 생성 모델

분류 모델은 입력 x가 어떤 레이블 y에 속하는지 결정함. 본 적 없는 데이터가 들어와도 기존 레이블 중 하나로 분류해버리는 한계가 있음.

생성 모델은 데이터 자체의 확률 분포 P(X)를 학습해서 새로운 데이터를 만들어냄. 비현실적인 입력은 확률이 거의 0이라고 판단함.


### 1-2. 생성 모델 분류

명시적 확률밀도 모델은 P(X)를 직접 수치로 출력할 수 있음.
- 정확히 계산 가능: 자기회귀 모델
- 근사적 계산: VAE

암시적 확률밀도 모델은 P(X)를 수치로는 못 알려주지만 샘플링은 가능함.
- 직접 샘플링: GAN
- 반복적 샘플링: 확산 모델


## 2. AE

입력과 동일한 출력을 만드는 비지도 학습 신경망임. 차원 축소, 특징 추출에 쓰임.

인코더는 입력 x를 잠재 벡터 z로 압축함.

잠재 벡터는 핵심 정보만 남긴 병목 구간임.

디코더는 z를 다시 x'로 복원함. 압축이 잘 됐는지 검증하는 역할임.

공간을 좁게 만드는 이유는 모델이 중요한 정보를 스스로 고민하게 만들기 위함임.


## 3. VAE

AE의 한계는 학습 안 된 빈 공간에서 뽑으면 디코더가 뜬금없는 걸 출력한다는 것. VAE는 잠재 벡터를 점이 아니라 가우시안 분포로 학습해서 해결.

잠재 공간 특징은 두 가지.
- 연속성: 가까운 두 점은 유사한 특징을 가짐
- 조밀함: 표준 정규 분포 주변에 모이도록 강제해서 빈 공간이 거의 없음

작동 방식은 3단게임.

1단계는 인코더가 뮤와 시그마를 계산함.

2단계는 그 범위에서 z를 샘플링함. 미분 불가능 문제는 재파라미터화 트릭으로 해결함. z = 뮤 + 시그마⊙엡실론

3단계는 디코더가 z로 x프라임을 생성함.


## 4. GAN

### 4-1. 구조

생성자는 노이즈 z를 입력받아 가짜 데이터를 만듦. 판별자를 속이는 게 목표.

판별자는 실제 데이터와 가짜 데이터를 입력받아 진짜일 확률을 출력함.

둘이 경쟁하면서 생성자는 진짜와 구분 불가능한 데이터를 만들게 되고, 판별자는 50% 확률로 찍는 수준에 도달하는 게 최적점.


### 4-2. 장단점

장점은 세 가지.
- 이미지가 선명함. VAE는 픽셀 평균을 최적화해서 흐릿해지는데 GAN은 판별자를 통과하면서 미세한 질감까지 살림
- 신경망 한 번 통과하면 결과물이 바로 나와서 빠름
- 잠재 공간 보간이 가능함

단점은 네 가지.
- 학습 불안정성. 손실 값이 요동치고 무한루프에 빠지기 쉬움
- 그래디언트 소멸. 판별자가 너무 완벽하면 생성자가 개선 방향을 못 찾음.
- 모드 붕괴. 특정 패턴 하나만 계속 만들어냄
- 역매핑 불가능. z에서 x로 가는 일방통행만 있음


## 5. 확산 모델

### 5-1. 구조

순확산은 원본 x0에 노이즈를 점진적으로 추가해서 완전한 노이즈 xT로 만드는 과정임.

역확산은 xT에서 시작해서 추가된 노이즈를 예측하면서 제거해 x0를 복원하는 과정이고 이걸 수십 번 반복함.


### 5-2. 장단점

장점은 세 가지.
- 단계마다 다듬기 때문에 GAN보다 품질이 높음
- MSE 손실 곡선을 확인할 수 있어서 학습 안정적임
- 모드 붕괴가 거의 없음

단점은 두 가지.
- 30~50번 이상 반복 계산이 필요해서 느림
- 노이즈와 데이터의 형태가 동일해야 하는 구조적 제약이 있음


## 6. 전이학습

### 6-1. 개념

처음부터 학습하면 데이터 부족, 계산 비용, 과적합 문제가 생김. 이미 학습된 모델의 지식을 새로운 문제에 활용하는 게 전이학습임.

도메인은 어떤 데이터를 다루는가임. 태스크는 그 데이터로 무엇을 예측하는가임.

소스는 이미 학습된 출발점. 타겟은 내가 풀고 싶은 도착점.

도메인이나 태스크 중 하나라도 다를 때 의미가 생김.


### 6-2. 분류

도메인과 태스크 상태에 따른 분류는 세 가지.
- 귀납적 전이학습: 태스크가 다르고 타겟에 라벨이 있음
- 트랜스듀서형 전이학습: 태스크는 같고 도메인이 다름. 타겟 라벨 없어도 됨
- 비지도 전이학습: 둘 다 다르고 라벨도 없음

무엇을 전이하느냐에 따른 분류는 네 가지.
- 인스턴스: 소스 데이터 중 유용한 샘플을 골라 가중치 부여
- 특징: 공통 특징 추출 방식을 전이
- 파라미터: 모델 가중치 자체를 가져옴. 현재 주류임
- 관계: 데이터 간 논리적 구조를 전이


### 6-3. Pre-training과 Fine-tuning

파운데이션 모델은 방대한 비정형 데이터로 비지도 학습한 거대 모델임.

파인튜닝 방식은 세 가지.
- 프로즌 백본: 전체 고정, 마지막 레이어만 교체
- 파셜 파인튜닝: 앞쪽 고정, 뒤쪽 일부만 학습
- 풀 파인튜닝: 전체 재학습. 학습률을 낮게 설정해야 함

PEFT는 원래 파라미터는 고정하고 작은 파라미터만 추가로 학습하는 방식.
- 어댑터: 레이어 사이에 소형 모듈 삽입
- LoRA: 가중치 변화량을 두 행렬 곱으로 표현. 현재 가장 널리 쓰임
- 프롬프트 튜닝: 입력 앞에 붙이는 소프트 프롬프트만 학습

제로샷은 학습 데이터 없이 바로 추론함.

퓨샷은 극소수 예시만으로 태스크를 수행함. 파라미터를 업데이트하지 않음.


### 6-4. 전이학습 한계

- 네거티브 트랜스퍼: 도메인이 너무 다르면 오히려 성능이 떨어짐
- 캐타스트로픽 포게팅: 파인튜닝 시 원래 능력을 잃어버림
- 바이어스 트랜스퍼: 사전 학습 데이터의 편향이 그대로 넘어옴
- 트랜스퍼빌리티 측정 어려움: 두 태스크의 관련성을 사전에 측정하기 어려움


## 7. AI Alignment

### 7-1. 개념

AI가 인간의 의도, 목표, 가치에 부합하도록 행동하게 만드는 연구 분야.

기준은 세 가지. 도움이 되는가, 정직한가, 무해한가.

아우터 얼라인먼트는 보상 기준 설계 문제임. 잘못 설계하면 AI가 꼼수로 보상을 극대화함.

이너 얼라인먼트는 모델 내부에서 실제로 뭘 최적화하는지의 문제임. 훈련 중엔 정렬된 척 하다가 배포 후엔 다른 목표를 수행할 수 있음.


### 7-2. RLHF

인간의 피드백을 기반으로 모델을 강화학습으로 조정하는 방법임. 정답 여부보다 인간이 선호하는 답변을 학습.

3단계로 진행됨.

1단계 SFT는 사람이 작성한 질문-답변 쌍으로 지도 학습.

2단계 보상 모델은 여러 답변에 사람이 순위를 매기면 선호도를 점수로 수치화하는 채점기를 만듦.

3단계 강화학습은 생성된 답변에 보상 모델이 점수를 매기고 모델이 높은 점수 방향으로 업데이트됨. PPO를 씀. KL 발산 제약으로 원래 지식을 잃지 않도록 규제.

주의점은 보상 해킹임. 모델이 인간이 좋아할 법한 대답만 골라 하는 기만적 행동이 나타날 수 있음.



## 8. 프롬프트 엔지니어링

### 8-1. 개념

AI가 더 좋은 답변을 내도록 입력을 설계하는 기술임. 모델은 그대로 두고 입력만 설계.



### 8-2. 주요 기법

- 제로샷: 예시 없이 바로 질문
- 퓨샷: 예시 몇 개 보여주고 질문. 맥락을 주기 때문에 정확도가 올라감
- CoT: 풀이 과정을 단계별로 생각하게 만듦. "Let's think step by step"만 붙여도 효과 있음
- 롤 프롬프팅: 구체적인 역할을 부여함
- 인스트럭션 프롬프팅: 형식, 길이, 톤을 구체적으로 지정함
- 셀프 컨시스턴시: 같은 질문을 여러 번 해서 가장 많은 답을 채택함
- RAG: 외부 데이터베이스에서 검색해서 프롬프트에 함께 넣어줌. 파인튜닝 없이 도메인 특화 답변이 가능함


### 8-3. 한계

- 프롬프트 인젝션: 악의적 사용자가 프롬프트를 조작해서 AI를 오작동시킬 수 있음
- 할루시네이션: 없는 정보를 지어내는 현상을 완전히 막기 어려움
- 모델 의존성: 모델마다 반응이 다르고 업데이트 시 기존 프롬프트가 안 먹힐 수 있음
- 성능 한계: 특화된 도메인에서는 파인튜닝이 필요할 수 있음